# Real-Time 3D Lower-Limb Analysis with MediaPipe

- Uses MediaPipe Pose to estimate body landmarks from webcam video
- Computes knee and hip angles using **3D landmark coordinates** instead of only 2D pixels
- Uses 2D pixel landmarks only for drawing overlays on the frame
- Draws a lower-limb skeleton, joint angle labels, ROM arc gauges, and trajectory trails
- Counts squat-like repetitions from knee-angle valleys
- Fires a posture alert when hip angle suggests forward lean
- Logs every frame to CSV and optionally records annotated output video


## 1. Import

In [ ]:
# Run once if packages are not installed:
# !pip install mediapipe==0.10.21 opencv-contrib-python==4.10.0.84 numpy==1.26.4 scipy==1.13.1 matplotlib==3.8.4 protobuf==4.25.3 ipython

import cv2
import mediapipe as mp
import numpy as np
import math
import time
import csv
import os
from collections import deque
from datetime import datetime
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Video, Markdown

print("All imports OK")

## 2. Init

In [ ]:
# Thresholds
VISIBILITY_THRESHOLD = 0.6
HIP_ANGLE_WARN = 40.0

# Visual
TRAIL_LENGTH = 30
ROM_GAUGE_RADIUS = 40

# Rep counting
REP_PEAK_PROMINENCE = 15.0
REP_PEAK_DISTANCE = 10

# Colours (BGR)
JOINT_COLOR = (0, 255, 0)
BONE_COLOR = (255, 165, 0)
ANGLE_COLOR = (255, 255, 255)
WARN_COLOR = (0, 0, 255)
TRAIL_COLOR = (0, 200, 255)

# CLAHE for frame pre-processing
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

print("set")

## 3. Utility Functions

In [ ]:
def preprocess_frame(frame: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = _clahe.apply(l)
    enhanced = cv2.merge([l, a, b])
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)


def landmark_to_px(landmark, frame_w: int, frame_h: int):
    return int(landmark.x * frame_w), int(landmark.y * frame_h)


def landmark_to_3d(landmark) -> np.ndarray:
    return np.array([landmark.x, landmark.y, landmark.z], dtype=float)


def compute_angle_2d(a, b, c) -> float:
    ba = np.array([a[0] - b[0], a[1] - b[1]], dtype=float)
    bc = np.array([c[0] - b[0], c[1] - b[1]], dtype=float)
    n_ba, n_bc = np.linalg.norm(ba), np.linalg.norm(bc)
    if n_ba == 0 or n_bc == 0:
        return 0.0
    cos_a = np.clip(np.dot(ba, bc) / (n_ba * n_bc), -1.0, 1.0)
    return math.degrees(math.acos(cos_a))


def compute_angle_3d(a, b, c) -> float:
    ba = np.array(a, dtype=float) - np.array(b, dtype=float)
    bc = np.array(c, dtype=float) - np.array(b, dtype=float)
    n_ba, n_bc = np.linalg.norm(ba), np.linalg.norm(bc)
    if n_ba == 0 or n_bc == 0:
        return 0.0
    cos_a = np.clip(np.dot(ba, bc) / (n_ba * n_bc), -1.0, 1.0)
    return math.degrees(math.acos(cos_a))


def compute_angle(a, b, c) -> float:
    # Backward-compatible name. In this notebook, angle math should call compute_angle_3d.
    return compute_angle_2d(a, b, c)


def detect_view_mode(lm, mp_pose) -> str:
    l_hip_x = lm[mp_pose.PoseLandmark.LEFT_HIP.value].x
    r_hip_x = lm[mp_pose.PoseLandmark.RIGHT_HIP.value].x
    return "side" if abs(l_hip_x - r_hip_x) < 0.06 else "front"


def _vis_ok(lm_item) -> bool:
    return lm_item.visibility >= VISIBILITY_THRESHOLD

print("Utility functions defined with 3D angle support")


## 4. Skeleton Drawing & ROM Gauge

In [ ]:
mp_pose_mod = mp.solutions.pose

LOWER_LIMB_CONNECTIONS = [
    (mp_pose_mod.PoseLandmark.LEFT_HIP, mp_pose_mod.PoseLandmark.LEFT_KNEE),
    (mp_pose_mod.PoseLandmark.LEFT_KNEE, mp_pose_mod.PoseLandmark.LEFT_ANKLE),
    (mp_pose_mod.PoseLandmark.RIGHT_HIP, mp_pose_mod.PoseLandmark.RIGHT_KNEE),
    (mp_pose_mod.PoseLandmark.RIGHT_KNEE, mp_pose_mod.PoseLandmark.RIGHT_ANKLE),
    (mp_pose_mod.PoseLandmark.LEFT_HIP, mp_pose_mod.PoseLandmark.RIGHT_HIP),
]

LOWER_JOINTS = {
    "L.Hip": mp_pose_mod.PoseLandmark.LEFT_HIP,
    "R.Hip": mp_pose_mod.PoseLandmark.RIGHT_HIP,
    "L.Knee": mp_pose_mod.PoseLandmark.LEFT_KNEE,
    "R.Knee": mp_pose_mod.PoseLandmark.RIGHT_KNEE,
    "L.Ankle": mp_pose_mod.PoseLandmark.LEFT_ANKLE,
    "R.Ankle": mp_pose_mod.PoseLandmark.RIGHT_ANKLE,
}


def draw_lower_limb_skeleton(frame, landmarks, frame_w, frame_h):
    lm = landmarks
    for start_lm, end_lm in LOWER_LIMB_CONNECTIONS:
        s_item = lm[start_lm.value]
        e_item = lm[end_lm.value]
        if not (_vis_ok(s_item) and _vis_ok(e_item)):
            continue
        start = landmark_to_px(s_item, frame_w, frame_h)
        end = landmark_to_px(e_item, frame_w, frame_h)
        cv2.line(frame, start, end, BONE_COLOR, 3, cv2.LINE_AA)

    for name, lm_id in LOWER_JOINTS.items():
        item = lm[lm_id.value]
        if not _vis_ok(item):
            continue
        px = landmark_to_px(item, frame_w, frame_h)
        cv2.circle(frame, px, 8, JOINT_COLOR, -1, cv2.LINE_AA)
        cv2.circle(frame, px, 9, (0, 0, 0), 1, cv2.LINE_AA)


def draw_rom_gauge(frame, center, current_angle, max_angle, label, side="right"):
    r = ROM_GAUGE_RADIUS
    start_angle = -90
    total_sweep = 180
    ox = center[0] + (r + 14) * (1 if side == "right" else -1)
    oy = center[1]

    if max_angle > 0:
        max_sweep = int(min(total_sweep * max_angle / 180.0, total_sweep))
        cv2.ellipse(frame, (ox, oy), (r, r), 0, start_angle, start_angle + max_sweep, (80, 80, 80), 6, cv2.LINE_AA)

    if current_angle > 0:
        cur_sweep = int(min(total_sweep * current_angle / 180.0, total_sweep))
        t = current_angle / 180.0
        arc_color = (int(50 + 205 * t), int(255 - 200 * t), 50)
        cv2.ellipse(frame, (ox, oy), (r, r), 0, start_angle, start_angle + cur_sweep, arc_color, 4, cv2.LINE_AA)

    cv2.putText(frame, f"{label}", (ox - 12, oy + r + 16), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (200, 200, 200), 1, cv2.LINE_AA)
    cv2.putText(frame, f"{current_angle:.0f} deg", (ox - 18, oy + 6), cv2.FONT_HERSHEY_SIMPLEX, 0.48, ANGLE_COLOR, 1, cv2.LINE_AA)

print("Skeleton & ROM drawing defined")

## 5. Trajectory Trail, Rep Counter, Posture Alert & Info Panel

In [ ]:
class TrajectoryTrail:
    def __init__(self, maxlen=TRAIL_LENGTH):
        self._buf = deque(maxlen=maxlen)

    def update(self, pt):
        self._buf.append(pt)

    def draw(self, frame):
        pts = list(self._buf)
        if len(pts) < 2:
            return
        for i in range(1, len(pts)):
            alpha = i / len(pts)
            color = (int(TRAIL_COLOR[0] * alpha), int(TRAIL_COLOR[1] * alpha), int(TRAIL_COLOR[2] * alpha))
            cv2.line(frame, pts[i - 1], pts[i], color, 2, cv2.LINE_AA)


class RepCounter:
    def __init__(self, history_size=120):
        self._history = deque(maxlen=history_size)
        self.reps = 0

    def update(self, angle, frame_idx):
        self._history.append(angle)
        if len(self._history) < 20:
            return self.reps
        arr = np.array(self._history)
        valleys, _ = find_peaks(-arr, prominence=REP_PEAK_PROMINENCE, distance=REP_PEAK_DISTANCE)
        self.reps = len(valleys)
        return self.reps


def draw_posture_alert(frame, hip_angle, threshold=HIP_ANGLE_WARN):
    if hip_angle > threshold:
        return
    h, w = frame.shape[:2]
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, h // 2 - 30), (w, h // 2 + 30), (0, 0, 200), -1)
    cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)
    cv2.putText(frame, "WARNING: FORWARD LEAN - STRAIGHTEN BACK", (w // 2 - 280, h // 2 + 10), cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 255, 50), 2, cv2.LINE_AA)


def draw_info_panel(frame, angles, fps, bg_mode, frame_idx, view_mode, reps_l, reps_r):
    panel_h, panel_w = 245, 330
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (panel_w, panel_h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    vm_col = (0, 220, 255) if view_mode == "side" else (180, 255, 180)
    lines = [
        ("AI Physiotherapy - MediaPipe v2", (0, 200, 255)),
        (f"Frame: {frame_idx:05d}   FPS: {fps:.1f}", (200, 200, 200)),
        (f"View: {view_mode.upper():<5}  BG: {'ON' if bg_mode else 'OFF'}", vm_col),
        ("", (255, 255, 255)),
        (f"L Knee  : {angles.get('L_knee', 0):.1f} deg   Reps: {reps_l}", (100, 255, 100)),
        (f"R Knee  : {angles.get('R_knee', 0):.1f} deg   Reps: {reps_r}", (100, 255, 100)),
        (f"L Hip   : {angles.get('L_hip', 0):.1f} deg", (255, 200, 100)),
        (f"R Hip   : {angles.get('R_hip', 0):.1f} deg", (255, 200, 100)),
    ]
    y = 22
    for text, color in lines:
        cv2.putText(frame, text, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.52, color, 1, cv2.LINE_AA)
        y += 26

print("Trail, RepCounter, Alerts, Panel defined")

## 6. Background Handler

In [ ]:
class BackgroundHandler:
    def __init__(self, history=120, var_threshold=40):
        self.subtractor = cv2.createBackgroundSubtractorMOG2(history=history, varThreshold=var_threshold, detectShadows=False)
        self.kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        self.kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    def process(self, frame):
        fg_mask = self.subtractor.apply(frame)
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, self.kernel_open)
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, self.kernel_close)
        fg_3ch = cv2.cvtColor(fg_mask, cv2.COLOR_GRAY2BGR)
        bg_mask = cv2.bitwise_not(fg_3ch)
        dimmed = cv2.addWeighted(frame, 0.25, np.zeros_like(frame), 0.75, 0)
        blended = np.where(bg_mask > 0, dimmed, frame)
        return fg_mask, blended.astype(np.uint8)

print("BackgroundHandler defined")

## 7. Real-Time Webcam Configuration

Set `VIDEO_SOURCE = 0` for your default webcam. If you have another camera, try `1` or `2`. Press **Q** or **Esc** in the OpenCV window to stop analysis.

In [ ]:
VIDEO_SOURCE = 0
CAMERA_WIDTH = 1280
CAMERA_HEIGHT = 720
CAMERA_FPS = 30

FLIP_FRAME = True          # mirror webcam for natural movement
BG_MODE = False            # optional background dimming
RECORD_OUTPUT = True       # save annotated live session

OUTPUT_DIR = r"C:\Users\sunil\OneDrive\Desktop"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "realtime_annotated_output_3d.mp4")
CSV_PATH = os.path.join(OUTPUT_DIR, "realtime_motion_log_3d.csv")

print("Real-time 3D configuration set")
print(f"CSV will save to: {CSV_PATH}")
print(f"Video will save to: {OUTPUT_PATH}")


## 8. Real-Time OpenCV Processing Loop

This cell opens your webcam and runs the full lower-limb analysis live. Stop it with **Q** or **Esc** in the OpenCV window.

In [ ]:
mp_pose = mp.solutions.pose

pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

cap = cv2.VideoCapture(VIDEO_SOURCE, cv2.CAP_DSHOW)
if not cap.isOpened():
    pose.close()
    raise RuntimeError(f"Cannot open camera source: {VIDEO_SOURCE}")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)
cap.set(cv2.CAP_PROP_FPS, CAMERA_FPS)

actual_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or CAMERA_WIDTH
actual_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or CAMERA_HEIGHT
actual_fps = cap.get(cv2.CAP_PROP_FPS) or CAMERA_FPS

os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)

writer = None
if RECORD_OUTPUT:
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, actual_fps, (actual_w, actual_h))
    if not writer.isOpened():
        writer.release()
        writer = None
        print(f"WARNING: Video writer could not open: {OUTPUT_PATH}")
        print("CSV logging will still continue. Try changing codec to 'XVID' or output extension to .avi if needed.")

bg_handler = BackgroundHandler()
trail_l = TrajectoryTrail()
trail_r = TrajectoryTrail()
rep_ctr_l = RepCounter()
rep_ctr_r = RepCounter()
max_rom = {"L_knee": 0.0, "R_knee": 0.0}

log_L_knee = []
log_R_knee = []
log_L_hip = []
log_R_hip = []
log_fps = []
log_view = []
log_reps_l = []
log_reps_r = []
log_rom_l = []
log_rom_r = []
log_posture = []
log_infer_ms = []

csv_fields = [
    "frame", "timestamp", "view_mode",
    "L_knee", "R_knee", "L_hip", "R_hip",
    "reps_L", "reps_R", "rom_max_L", "rom_max_R",
    "posture_alert", "infer_ms",
]

csv_file = open(CSV_PATH, "w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
csv_writer.writeheader()
print(f"Writing CSV to: {os.path.abspath(CSV_PATH)}")
if writer is not None:
    print(f"Writing annotated video to: {os.path.abspath(OUTPUT_PATH)}")

frame_idx = 0
prev_time = time.time()
window_name = "AI Physiotherapy - Real-Time Lower Limb Analysis"

print("Press Q or Esc")

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            print("Camera frame not available; stopping.")
            break

        if FLIP_FRAME:
            frame = cv2.flip(frame, 1)

        frame_idx += 1
        frame_h, frame_w = frame.shape[:2]

        curr_time = time.time()
        fps = 1.0 / max(curr_time - prev_time, 1e-6)
        prev_time = curr_time

        if BG_MODE:
            _, frame = bg_handler.process(frame)
        else:
            bg_handler.subtractor.apply(frame)

        proc = preprocess_frame(frame)
        t0 = time.time()
        rgb = cv2.cvtColor(proc, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = pose.process(rgb)
        infer_ms = (time.time() - t0) * 1000.0

        angles = {"L_knee": 0.0, "R_knee": 0.0, "L_hip": 0.0, "R_hip": 0.0}
        posture_ok = True
        view_mode = "N/A"

        if results.pose_landmarks:
            lm = results.pose_landmarks.landmark
            world_lm = results.pose_world_landmarks.landmark if results.pose_world_landmarks else None
            view_mode = detect_view_mode(lm, mp_pose)
            draw_lower_limb_skeleton(frame, lm, frame_w, frame_h)

            def get_px(lm_id):
                item = lm[lm_id.value]
                if _vis_ok(item):
                    return landmark_to_px(item, frame_w, frame_h)
                return None

            def get_3d(lm_id):
                item_2d = lm[lm_id.value]
                if not _vis_ok(item_2d):
                    return None
                if world_lm is not None:
                    return landmark_to_3d(world_lm[lm_id.value])
                return landmark_to_3d(item_2d)

            L_hip_px = get_px(mp_pose.PoseLandmark.LEFT_HIP)
            R_hip_px = get_px(mp_pose.PoseLandmark.RIGHT_HIP)
            L_knee_px = get_px(mp_pose.PoseLandmark.LEFT_KNEE)
            R_knee_px = get_px(mp_pose.PoseLandmark.RIGHT_KNEE)
            L_ankle_px = get_px(mp_pose.PoseLandmark.LEFT_ANKLE)
            R_ankle_px = get_px(mp_pose.PoseLandmark.RIGHT_ANKLE)
            L_shldr_px = get_px(mp_pose.PoseLandmark.LEFT_SHOULDER)
            R_shldr_px = get_px(mp_pose.PoseLandmark.RIGHT_SHOULDER)

            L_hip_3d = get_3d(mp_pose.PoseLandmark.LEFT_HIP)
            R_hip_3d = get_3d(mp_pose.PoseLandmark.RIGHT_HIP)
            L_knee_3d = get_3d(mp_pose.PoseLandmark.LEFT_KNEE)
            R_knee_3d = get_3d(mp_pose.PoseLandmark.RIGHT_KNEE)
            L_ankle_3d = get_3d(mp_pose.PoseLandmark.LEFT_ANKLE)
            R_ankle_3d = get_3d(mp_pose.PoseLandmark.RIGHT_ANKLE)
            L_shldr_3d = get_3d(mp_pose.PoseLandmark.LEFT_SHOULDER)
            R_shldr_3d = get_3d(mp_pose.PoseLandmark.RIGHT_SHOULDER)

            if L_knee_3d is not None and L_hip_3d is not None and L_ankle_3d is not None:
                angles["L_knee"] = compute_angle_3d(L_hip_3d, L_knee_3d, L_ankle_3d)
            if R_knee_3d is not None and R_hip_3d is not None and R_ankle_3d is not None:
                angles["R_knee"] = compute_angle_3d(R_hip_3d, R_knee_3d, R_ankle_3d)
            if L_hip_3d is not None and L_shldr_3d is not None and L_knee_3d is not None:
                angles["L_hip"] = compute_angle_3d(L_shldr_3d, L_hip_3d, L_knee_3d)
            if R_hip_3d is not None and R_shldr_3d is not None and R_knee_3d is not None:
                angles["R_hip"] = compute_angle_3d(R_shldr_3d, R_hip_3d, R_knee_3d)

            max_rom["L_knee"] = max(max_rom["L_knee"], angles["L_knee"])
            max_rom["R_knee"] = max(max_rom["R_knee"], angles["R_knee"])

            if L_knee_px:
                cv2.putText(frame, f"LK:{angles['L_knee']:.0f}", (L_knee_px[0] + 12, L_knee_px[1] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (100, 255, 100), 2, cv2.LINE_AA)
            if R_knee_px:
                cv2.putText(frame, f"RK:{angles['R_knee']:.0f}", (R_knee_px[0] + 12, R_knee_px[1] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (100, 255, 100), 2, cv2.LINE_AA)
            if L_hip_px:
                cv2.putText(frame, f"LH:{angles['L_hip']:.0f}", (L_hip_px[0] + 12, L_hip_px[1] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 200, 100), 2, cv2.LINE_AA)
            if R_hip_px:
                cv2.putText(frame, f"RH:{angles['R_hip']:.0f}", (R_hip_px[0] + 12, R_hip_px[1] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 200, 100), 2, cv2.LINE_AA)

            if L_knee_px:
                trail_l.update(L_knee_px)
            if R_knee_px:
                trail_r.update(R_knee_px)
            trail_l.draw(frame)
            trail_r.draw(frame)

            if L_knee_px:
                draw_rom_gauge(frame, L_knee_px, angles["L_knee"], max_rom["L_knee"], "L-ROM", side="left")
            if R_knee_px:
                draw_rom_gauge(frame, R_knee_px, angles["R_knee"], max_rom["R_knee"], "R-ROM", side="right")

            hip_values = [v for v in (angles["L_hip"], angles["R_hip"]) if v > 0]
            if hip_values:
                min_hip = min(hip_values)
                if min_hip <= HIP_ANGLE_WARN:
                    posture_ok = False
                draw_posture_alert(frame, min_hip)

        reps_l = rep_ctr_l.update(angles["L_knee"], frame_idx)
        reps_r = rep_ctr_r.update(angles["R_knee"], frame_idx)

        draw_info_panel(frame, angles, fps, BG_MODE, frame_idx, view_mode, reps_l, reps_r)

        log_L_knee.append(angles["L_knee"])
        log_R_knee.append(angles["R_knee"])
        log_L_hip.append(angles["L_hip"])
        log_R_hip.append(angles["R_hip"])
        log_fps.append(fps)
        log_view.append(view_mode)
        log_reps_l.append(reps_l)
        log_reps_r.append(reps_r)
        log_rom_l.append(max_rom["L_knee"])
        log_rom_r.append(max_rom["R_knee"])
        log_posture.append(0 if posture_ok else 1)
        log_infer_ms.append(infer_ms)

        csv_writer.writerow({
            "frame": frame_idx,
            "timestamp": datetime.now().isoformat(timespec="milliseconds"),
            "view_mode": view_mode,
            "L_knee": round(angles["L_knee"], 2),
            "R_knee": round(angles["R_knee"], 2),
            "L_hip": round(angles["L_hip"], 2),
            "R_hip": round(angles["R_hip"], 2),
            "reps_L": reps_l,
            "reps_R": reps_r,
            "rom_max_L": round(max_rom["L_knee"], 2),
            "rom_max_R": round(max_rom["R_knee"], 2),
            "posture_alert": 0 if posture_ok else 1,
            "infer_ms": round(infer_ms, 2),
        })

        if writer is not None:
            writer.write(frame)

        cv2.imshow(window_name, frame)
        key = cv2.waitKey(1) & 0xFF
        if key in (ord("q"), 27):
            break

finally:
    csv_file.close()
    cap.release()
    if writer is not None:
        writer.release()
    pose.close()
    cv2.destroyAllWindows()

print(f"\nReal-time analysis complete - {frame_idx} frames processed")
print(f"CSV saved -> {os.path.abspath(CSV_PATH)}")
if RECORD_OUTPUT:
    print(f"Annotated video saved -> {os.path.abspath(OUTPUT_PATH)}")
if frame_idx > 0:
    print(f"Final reps  L={log_reps_l[-1]}  R={log_reps_r[-1]}")
    print(f"Max ROM     L={max_rom['L_knee']:.1f} deg  R={max_rom['R_knee']:.1f} deg")

## 9. Post-Session Plots

Run this after stopping the real-time loop to review knee angles, rep detections, posture alerts, and inference speed.

In [ ]:
if not log_L_knee:
    print("No data found")
else:
    frame_nums = np.arange(1, len(log_L_knee) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(15, 9))

    axes[0, 0].plot(frame_nums, log_L_knee, label="Left Knee", linewidth=1.2)
    axes[0, 0].plot(frame_nums, log_R_knee, label="Right Knee", linewidth=1.2)
    axes[0, 0].set_title("Knee Angle Over Time")
    axes[0, 0].set_xlabel("Frame")
    axes[0, 0].set_ylabel("Angle (deg)")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    for data, label, color in [
        (log_L_knee, "Left Reps", "tab:blue"),
        (log_R_knee, "Right Reps", "tab:green"),
    ]:
        arr = np.array(data)
        valleys, _ = find_peaks(-arr, prominence=REP_PEAK_PROMINENCE, distance=REP_PEAK_DISTANCE)
        axes[0, 1].plot(frame_nums, arr, label=label, color=color, linewidth=1)
        axes[0, 1].plot(frame_nums[valleys], arr[valleys], "rv", markersize=6)
    axes[0, 1].set_title("Rep Detection")
    axes[0, 1].set_xlabel("Frame")
    axes[0, 1].set_ylabel("Angle (deg)")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(frame_nums, log_infer_ms, linewidth=0.8)
    axes[1, 0].axhline(np.mean(log_infer_ms), color="red", linestyle="--", label=f"Mean: {np.mean(log_infer_ms):.1f} ms")
    axes[1, 0].set_title("Inference Time")
    axes[1, 0].set_xlabel("Frame")
    axes[1, 0].set_ylabel("ms")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].fill_between(frame_nums, log_posture, step="mid", color="red", alpha=0.45)
    axes[1, 1].set_title("Posture Alert Timeline")
    axes[1, 1].set_xlabel("Frame")
    axes[1, 1].set_yticks([0, 1])
    axes[1, 1].set_yticklabels(["OK", "WARN"])
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    summary = f"""
| Metric | Left | Right |
|--------|------|-------|
| **Final Rep Count** | {log_reps_l[-1]} | {log_reps_r[-1]} |
| **Max ROM (deg)** | {log_rom_l[-1]:.1f} | {log_rom_r[-1]:.1f} |
| **Mean Knee Angle (deg)** | {np.mean(log_L_knee):.1f} | {np.mean(log_R_knee):.1f} |
| **Knee Angle Std Dev (deg)** | {np.std(log_L_knee):.2f} | {np.std(log_R_knee):.2f} |
| **Posture Alerts** | {sum(log_posture)} frames | {100*sum(log_posture)/len(log_posture):.1f}% |
| **Mean Inference (ms)** | {np.mean(log_infer_ms):.1f} | - |
| **Total Frames** | {len(log_L_knee)} | - |
"""
    display(Markdown(summary))

## 10. View Recorded Output

In [ ]:
if RECORD_OUTPUT and os.path.exists(OUTPUT_PATH):
    print(f"Recorded output: {os.path.abspath(OUTPUT_PATH)}")
    display(Video(OUTPUT_PATH, embed=True))
else:
    print("No recorded video found. Set RECORD_OUTPUT=True and run the real-time loop.")